# 03 · Embeddings, similarity, visualization

**Agenda: 40–55 min.** Every frame has a CLIP embedding (`clip`), a similarity index (`frames_sim`) and a 2-D UMAP layout (`frames_viz`). The presenter computes these on Nebius for the full deployment; here they are precomputed so the workflow stays interactive on a laptop.

In [ ]:
import fiftyone as fo
from fiftyone import ViewField as F

frames = fo.load_dataset("droid-frames-workshop")
print(frames.list_brain_runs())
session = fo.launch_app(frames)

## Embeddings panel

Open the **Embeddings** panel (the `+` next to Samples), choose `frames_viz`.

1. **Color by `camera`** — three clusters: wrist, ext1, ext2. The wrist camera lives in its own world.
2. **Color by `task_type`** — within each camera cluster, fold-cloth / clump / brick tasks separate.
3. **Color by `gripper_open`** — does the embedding know when something is being held?
4. **Lasso** a small island far from everything → the grid shows what it is. Usually a human in frame, a dropped object, or a camera knocked out of place.

## Similarity search

Pick one frame in the grid, click the image-search icon (or use the code below) → the grid re-sorts by similarity. This is how you find *more of a failure* once you have seen one.

In [ ]:
# The frame with the highest uniqueness: find its neighbours
odd = frames.sort_by("uniqueness", reverse=True).first()
similar = frames.sort_by_similarity(odd.id, k=50, brain_key="frames_sim")
session.view = similar

## Text search against the same index

The index was built with CLIP, so it also takes text prompts.

In [ ]:
for prompt in ["a robot gripper holding a blue brick", "a person's hand", "an open drawer"]:
    view = frames.sort_by_similarity(prompt, k=24, brain_key="frames_sim")
    print(prompt, "→", view.first().episode_id, view.first().camera)

session.view = frames.sort_by_similarity("a robot gripper holding a blue brick", k=48, brain_key="frames_sim")

## Optional: compute embeddings yourself (≈1–2 min on CPU for 200 frames)

In [ ]:
# OPTIONAL — 200 frames, CPU
# import fiftyone.brain as fob
# import fiftyone.zoo as foz
# slice_ = frames.take(200, seed=51)
# model = foz.load_zoo_model("clip-vit-base32-torch")
# slice_.compute_embeddings(model, embeddings_field="clip_mine", num_workers=0)
# fob.compute_visualization(slice_, embeddings="clip_mine", brain_key="mine_viz", method="umap")
# session.view = slice_

### Where this ran for the presenter

The same `compute_embeddings` call, on the 100-episode deployment, runs as a **Nebius Serverless AI Job**: a GPU container that reads the frames from object storage and writes vectors back. The interactive part — lasso, sort, tag — stays on the laptop.

**Next:** we have frames worth labeling. Notebook 04 labels them without a human drawing a box.